In [ ]:
# Adjust lr and betas based on magnitude of loss
# save trained model at several tol levels specified in tol_v


In [1]:

# For the NN structure, we need the parameters below.
# d_in: dimension of input
# d_out: dimension of output
# layers: number of hidden layers
# nodes: number of nodes in each hidden layers
# We also need to specify activation: the activation function
#
# For the AdamW optimizer, we need the hyperparameters.
# lr, eps, betas, wd
#
# For NN training, we need 
# epoch0: starting epoch
# epochs: maximum number of NEW epochs allowed
# tol_v: LIST of tolerances for loss (MSE)
# seg_size: print out info every "seg_size" epochs;
# m1, m2, eta, eta_min
#     if loss is not reduced by eta for "m1*seg_size" epochs, reduce eta;
#     if eta < eta_min, stop training;
#     if loss is not improving for "m2*seg_size" epochs, stop training.
# tol_Lv, lr_new, betas_new
#     if tol <= tol_Lv[kp], update lr = lr_new[kp]; betas = betas_new[kp]
#
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt 
import math, time, os

torch.set_default_dtype(torch.float64)
activation = F.sigmoid # F.silu


In [2]:

# define Net()
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.layers = nn.ModuleList()
        self.layers.append(nn.Linear(d_in, nodes))  # First layer
        for _ in range(1, layers):  # Remaining hidden layers
            self.layers.append(nn.Linear(nodes, nodes))
        self.out = nn.Linear(nodes, d_out)
    def forward(self, x):
        for layer in self.layers:
            x = activation(layer(x)) # Or use torch.relu
        return self.out(x)

# build the NN
def build_nn():
    model = Net()
    optimizer = optim.AdamW(model.parameters(), lr=lr, eps=eps,
        betas=betas,weight_decay=wd)
    return model, optimizer

# rebuild the NN from (model_dict, optimizer_dict) 
def rebuild_nn():
    model = Net()
    optimizer = optim.AdamW(model.parameters())
    if 'model_dict' in globals():
        model.load_state_dict(model_dict)
    else:
        print('**** Not enough info to recover the model ****')
    if 'optimizer_dict' in globals():
        optimizer.load_state_dict(optimizer_dict)
    else:
        print('**** Not enough info to recover the optimizer ****')
    return model, optimizer


In [3]:
# print parameters, model, optimer
def print_model_optimizer():
    print('*** Structure of neural network: ',model) # print the model
    print('*** Optimizer:  ',optimizer) # print the optimizer

def print_params():
    print('*** Parameters used:  ')
    for name in params_def:
        print(f'{name}: {params_def[name]}')

# save model and key variables to a data file
def save_model_vars(loss_v, loss_t1_v, loss_t2_v, out_file):
    pt_vars = {name: val for name, val in globals().items()
            if isinstance(val, (float, int))}
    pt_vars['model_dict'] = model.state_dict()
    pt_vars['optimizer_dict'] = optimizer.state_dict()
    pt_vars['loss_v'] = torch.tensor(loss_v)
    pt_vars['loss_t1_v'] = torch.tensor(loss_t1_v)
    pt_vars['loss_t2_v'] = torch.tensor(loss_t2_v)
    pt_vars['epoch_0']=epoch_0+len(loss_v)-1
    torch.save(pt_vars, out_file)
    # print(pt_vars.keys())


In [4]:
# train the NN
def train_nn():
    model.train();
    optimizer.zero_grad() # zero the gradients
    loss = F.mse_loss(model(xtrain),ytrain) # train loss
    loss_cur = loss.item();
    loss_saved = loss_cur; epoch_saved = -1
    loss_min = loss_cur;
    loss_t1_min = 10; loss_t2_min = 10; 
    # We start the for-loop with loss already calculated!
    
    eta = 10; kp = 0; tol_L = tol_Lv[kp]
    k_tol = 0; tol = tol_v[k_tol]; out_file=f'{out_base}_{tol}.pt'
    start = time.time()
    loss_v = []
    loss_t1_v = []; loss_t2_v = []
        # loss_v[i] stores loss AFTER completing epoch [i]
    for i in range(epochs):
        # Back propagation to compute all gradients
        loss.backward()
        
        # update lr, betas before optimizer step if triggered by loss 
        if loss_cur <= tol_L:
            for g in optimizer.param_groups:
                g['lr'] = lr_new[kp]
                g['betas'] = betas_new[kp]
            print(f"** optimizer at i = {i}: lr = {g['lr']}, betas = {g['betas']}")
            kp += 1
            tol_L = tol_Lv[kp] if kp < len(tol_Lv) else -1
        optimizer.step() # update step in epoch i

        # Calculate loss after the update in epoch i
        optimizer.zero_grad() # zero the gradients
        loss = F.mse_loss(model(xtrain),ytrain) # train loss
        loss_cur = loss.item()
        loss_v.append(loss_cur)
        loss_min = min(loss_min, loss_cur)
        with torch.no_grad(): # no need for grad in loss_test!!!
            loss_t1_cur = F.mse_loss(model(xt1), yt1).item() # test loss
            loss_t2_cur = F.mse_loss(model(xt2), yt2).item()
        loss_t1_v.append(loss_t1_cur)
        loss_t1_min = min(loss_t1_min, loss_t1_cur)
        loss_t2_v.append(loss_t2_cur)
        loss_t2_min = min(loss_t2_min, loss_t2_cur)
        
        # print out info every seg_size epochs
        if (i+epoch_0)%seg_size == 0:
            print(f'i = {i+epoch_0}, loss_saved = {loss_saved:.4e}, eta = {eta:.4f}, '
                  f'time = {time.strftime("%H:%M:%S")}')
            print(f'  loss = {loss_cur:.4e}, loss_t1 = {loss_t1_cur:.4e}, loss_t2 = {loss_t2_cur:.4e}')
            print(f'  L_min = {loss_min:.4e}, L_t1min = {loss_t1_min:.4e}, L_t2min = {loss_t2_min:.4e}')
        
        # if loss is not reduced by eta for "m1*seg_size" epochs, reduce eta
        if i == epoch_saved+m1*seg_size:
            eta = 1+0.95*(loss_saved/loss_min-1)

        # if loss is reduced by eta, save a copy of the model
        if loss_cur <= loss_saved/eta or loss_cur <= tol:
            os.sys.stderr.write(f'save: i={i}, loss={loss_cur:.4e}, eta={eta:.4f}, tol={tol}, time = {time.strftime("%H:%M:%S")}')
            os.sys.stderr.write(f'\n loss = {loss_cur:.4e}, loss_t1 = {loss_t1_cur:.4e}, loss_t2 = {loss_t2_cur:.4e}')
            save_model_vars(loss_v, loss_t1_v, loss_t2_v, out_file)
            loss_saved = loss_cur; epoch_saved = i
        if loss_cur <= tol:
            k_tol += 1
            if k_tol >= len(tol_v): break
            tol = tol_v[k_tol]
            out_file=f'{out_base}_{tol}.pt'
        
        # if loss is not reduced by eta for "m2*seg_size" epochs, stop training
        if i == epoch_saved+m2*seg_size or eta < eta_min:
            break
        
    # print more info after training
    current = time.time();
    print(f'>> Stopped at i = {i}, loss_cur = {loss_cur:.4e}, eta = {eta:.4f}, tol={tol}')
    print(f'>> Elapsed time = {current-start:.2f}s, CA time = {time.ctime(current):s}')
    msg_train=[]
    msg_train.append(f'Training info: loss_saved = {loss_saved:.4e}, '\
            f'epoch_cumul = {epoch_0+epoch_saved}')
    msg_train.append(f'    loss_min = {loss_min:.4e}, eta = {eta:.4f}')
    msg_train.append(f'    Elapsed time = {current-start:.2f}s,  '\
            f'CA time = {time.ctime(current):s}')
    return {'loss_v':torch.tensor(loss_v),
            'loss_t1_v':torch.tensor(loss_t1_v),
            'loss_t2_v':torch.tensor(loss_t2_v),
            'msg_train':msg_train}
